# Real vs generated pulse: LivePortrait self-reenactment

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. When asked, upload `pilot/upload/source.png` and `pilot/upload/driving.mp4`.
3. The last cell downloads the generated video. Save it as `pilot/eval/generated/vid.mp4` and run `python rppg_demo.py --eval pilot/eval` (PyCharm button 7).

Model: LivePortrait (Guo et al., 2024, https://github.com/KwaiVGI/LivePortrait). Data: UBFC-rPPG subject 1 (Bobbia et al., 2017). The files live only on this temporary VM.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

Clone to the VM disk (not Google Drive: it is much slower). If a later cell says "No module named X", run `!pip install -q X` and re-run that cell.

In [ ]:
!git clone -q https://github.com/KwaiVGI/LivePortrait.git /content/LivePortrait
%cd /content/LivePortrait
# Keep Colab's own torch/numpy/opencv; install everything else without version pins,
# so pip downloads ready-made packages instead of compiling old ones.
!cat requirements*.txt | grep -v '^-r' | grep -v '^#' | sed 's/[=<>~;].*//' | tr -d ' ' | grep -v '^$' \n  | grep -viE '^(torch|torchvision|torchaudio|numpy|opencv-python|opencv-python-headless|onnxruntime|onnxruntime-gpu)$' \n  | sort -u > /tmp/req.txt
!echo "Installing:" && cat /tmp/req.txt
!pip install -q -r /tmp/req.txt onnxruntime-gpu

In [ ]:
%cd /content/LivePortrait
from huggingface_hub import snapshot_download
snapshot_download('KwaiVGI/LivePortrait', local_dir='pretrained_weights',
                  ignore_patterns=['*.git*', 'README.md', 'docs/*'])
!ls pretrained_weights

In [ ]:
%cd /content/LivePortrait
from google.colab import files
uploaded = files.upload()   # choose source.png and driving.mp4
print(list(uploaded))

Self-reenactment: the still photo (first frame) is animated with the motion of the same video. The face region of every output frame is generated; the background is pasted back from the photo.

In [ ]:
%cd /content/LivePortrait
!python inference.py -s source.png -d driving.mp4 --flag_crop_driving_video

In [ ]:
import glob, cv2
outs = sorted(glob.glob('animations/*.mp4'))
print(outs)
gen = [p for p in outs if not p.endswith('_concat.mp4')][0]
c = cv2.VideoCapture(gen)
print(gen, int(c.get(cv2.CAP_PROP_FRAME_COUNT)), 'frames at', c.get(cv2.CAP_PROP_FPS), 'fps (driving: 878 frames)')
files.download(gen)
concat = [p for p in outs if p.endswith('_concat.mp4')]
if concat: files.download(concat[0])   # side-by-side, useful for the slide